In [1]:
# Install dependencies
import subprocess
result = subprocess.run(
    ["pip", "install", "-r", "/workspace/shared/audit_validator/requirements.txt", "-q"],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if result.stdout else "")
print(result.stderr[-1000:] if result.returncode != 0 else "✅ All packages installed")


✅ All packages installed


In [2]:
# Verify ROCm + PyTorch on AMD GPU
import torch

print("=== GPU / ROCm Check ===")
print(f"PyTorch version     : {torch.__version__}")
print(f"CUDA available      : {torch.cuda.is_available()}")  # ROCm shows as CUDA
print(f"Number of GPUs      : {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"\nGPU {i}: {props.name}")
        print(f"  Total memory : {props.total_memory / 1e9:.2f} GB")
    print("\n✅ GPU is ready!")
else:
    print("⚠️ No GPU found — check session")

=== GPU / ROCm Check ===
PyTorch version     : 2.10.0+rocm7.2.4.git3d3aa833
CUDA available      : True
Number of GPUs      : 1

GPU 0: AMD Instinct MI300X
  Total memory : 206.14 GB

✅ GPU is ready!


In [3]:
# Quick tensor test on GPU
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

x = torch.randn(1000, 1000).to(device)
y = torch.mm(x, x.T)
print(f"Matrix multiply result shape: {y.shape}")
print("✅ GPU tensor ops working")

Using device: cuda
Matrix multiply result shape: torch.Size([1000, 1000])
✅ GPU tensor ops working


In [4]:
# GPU memory logger
import torch

def log_gpu_memory(label=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved  = torch.cuda.memory_reserved() / 1e9
        total     = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"[GPU Memory] {label}")
        print(f"  Allocated : {allocated:.2f} GB")
        print(f"  Reserved  : {reserved:.2f} GB")
        print(f"  Total     : {total:.2f} GB")
        print(f"  Free      : {total - reserved:.2f} GB")

log_gpu_memory("Baseline — before loading anything")

[GPU Memory] Baseline — before loading anything
  Allocated : 0.09 GB
  Reserved  : 0.10 GB
  Total     : 206.14 GB
  Free      : 206.04 GB


In [5]:
# Cell 5 — Create compliance rules
import json, os

RULES_DIR = "/workspace/shared/audit_validator/data/compliance_rules"
os.makedirs(RULES_DIR, exist_ok=True)

# GDPR Rules
gdpr_rules = {
  "framework": "GDPR",
  "version": "2018",
  "rules": [
    {
      "id": "GDPR-001",
      "category": "Data Collection",
      "title": "Lawful basis for processing",
      "description": "Personal data must be processed on a lawful basis such as consent, contract, legal obligation, vital interests, public task, or legitimate interests.",
      "keywords": ["consent", "lawful basis", "processing", "personal data"],
      "severity": "HIGH",
      "required_clause": "The document must state the legal basis under which personal data is collected and processed."
    },
    {
      "id": "GDPR-002",
      "category": "Data Retention",
      "title": "Retention period",
      "description": "Personal data must not be kept longer than necessary for its original purpose.",
      "keywords": ["retention", "storage period", "deletion", "data kept"],
      "severity": "HIGH",
      "required_clause": "A clear data retention period or deletion schedule must be defined."
    },
    {
      "id": "GDPR-003",
      "category": "Data Subject Rights",
      "title": "Right to access and erasure",
      "description": "Data subjects have the right to access their personal data and request erasure.",
      "keywords": ["right to access", "erasure", "right to be forgotten", "subject request"],
      "severity": "HIGH",
      "required_clause": "Procedures for handling data subject access and erasure requests must be documented."
    },
    {
      "id": "GDPR-004",
      "category": "Data Breach",
      "title": "Breach notification",
      "description": "Data breaches must be reported to supervisory authority within 72 hours.",
      "keywords": ["breach", "notification", "72 hours", "incident"],
      "severity": "CRITICAL",
      "required_clause": "A breach notification procedure with 72-hour reporting timeline must be included."
    },
    {
      "id": "GDPR-005",
      "category": "Third Party",
      "title": "Data processor agreements",
      "description": "Contracts with third-party data processors must include GDPR-compliant clauses.",
      "keywords": ["processor", "third party", "data sharing", "sub-processor"],
      "severity": "MEDIUM",
      "required_clause": "Data processing agreements with all third parties must be referenced."
    }
  ]
}

# SOX Rules
sox_rules = {
  "framework": "SOX",
  "version": "Sarbanes-Oxley Act 2002",
  "rules": [
    {
      "id": "SOX-001",
      "category": "Internal Controls",
      "title": "Internal control over financial reporting",
      "description": "Management must assess the effectiveness of internal controls over financial reporting.",
      "keywords": ["internal control", "financial reporting", "ICFR", "assessment"],
      "severity": "CRITICAL",
      "required_clause": "The document must describe internal controls over financial reporting."
    },
    {
      "id": "SOX-002",
      "category": "Audit",
      "title": "Independent auditor attestation",
      "description": "External auditors must attest to management's assessment of internal controls.",
      "keywords": ["auditor", "attestation", "external audit", "independent"],
      "severity": "HIGH",
      "required_clause": "Reference to independent auditor review of internal controls is required."
    },
    {
      "id": "SOX-003",
      "category": "Record Retention",
      "title": "Financial record retention",
      "description": "All financial records and audit workpapers must be retained for at least 7 years.",
      "keywords": ["record retention", "7 years", "financial records", "workpapers"],
      "severity": "HIGH",
      "required_clause": "A 7-year financial record retention policy must be stated."
    },
    {
      "id": "SOX-004",
      "category": "Fraud Prevention",
      "title": "Whistleblower protections",
      "description": "Employees must be protected from retaliation when reporting suspected fraud.",
      "keywords": ["whistleblower", "fraud", "retaliation", "reporting"],
      "severity": "HIGH",
      "required_clause": "Whistleblower protection policy must be documented."
    }
  ]
}

# Insurance / Financial Services Rules
insurance_rules = {
  "framework": "Insurance_Compliance",
  "version": "General Financial Services",
  "rules": [
    {
      "id": "INS-001",
      "category": "Policy Disclosure",
      "title": "Clear policy terms disclosure",
      "description": "All policy terms, exclusions, and conditions must be clearly disclosed to policyholders.",
      "keywords": ["disclosure", "policy terms", "exclusions", "conditions"],
      "severity": "HIGH",
      "required_clause": "All exclusions and conditions must be explicitly listed."
    },
    {
      "id": "INS-002",
      "category": "Claims",
      "title": "Claims processing timeline",
      "description": "Claims must be acknowledged within 10 business days and resolved within 30 days.",
      "keywords": ["claims", "processing", "timeline", "acknowledgment", "settlement"],
      "severity": "MEDIUM",
      "required_clause": "Claims processing timelines must be specified."
    },
    {
      "id": "INS-003",
      "category": "Anti-Money Laundering",
      "title": "AML/KYC compliance",
      "description": "Customer identity verification and AML checks must be performed.",
      "keywords": ["AML", "KYC", "identity verification", "money laundering"],
      "severity": "CRITICAL",
      "required_clause": "AML and KYC procedures must be documented."
    },
    {
      "id": "INS-004",
      "category": "Solvency",
      "title": "Capital adequacy requirements",
      "description": "Minimum capital reserves must be maintained as per regulatory requirements.",
      "keywords": ["capital", "solvency", "reserves", "adequacy"],
      "severity": "CRITICAL",
      "required_clause": "Capital adequacy and solvency margin must be stated."
    }
  ]
}

# Save all rule files
for name, data in [("gdpr_rules", gdpr_rules), ("sox_rules", sox_rules), ("insurance_rules", insurance_rules)]:
    path = f"{RULES_DIR}/{name}.json"
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    print(f"✅ Created: {path} ({len(data['rules'])} rules)")

print("\n📁 Rules knowledge base ready!")

✅ Created: /workspace/shared/audit_validator/data/compliance_rules/gdpr_rules.json (5 rules)
✅ Created: /workspace/shared/audit_validator/data/compliance_rules/sox_rules.json (4 rules)
✅ Created: /workspace/shared/audit_validator/data/compliance_rules/insurance_rules.json (4 rules)

📁 Rules knowledge base ready!


In [6]:
# Cell 6 — Test document parser
import sys
sys.path.insert(0, "/workspace/shared/audit_validator")
from src.document_parser import parse_document, chunk_document

# Create a simple test text document
sample_text = """
DATA PROCESSING AGREEMENT

1. LAWFUL BASIS FOR PROCESSING
The Company processes personal data under the lawful basis of legitimate interests
and contractual necessity as defined under Article 6 of GDPR.

2. DATA RETENTION
Customer data will be retained for a period of 3 years from the date of last transaction,
after which it will be securely deleted in accordance with our deletion policy.

3. BREACH NOTIFICATION
In the event of a personal data breach, the Company will notify the relevant supervisory
authority within 72 hours of becoming aware of the breach.

4. INTERNAL CONTROLS
Management conducts quarterly assessments of internal controls over financial reporting
in compliance with SOX requirements.
"""

# Save as test doc
with open("/workspace/shared/audit_validator/data/sample_docs/sample_contract.txt", "w") as f:
    f.write(sample_text)

doc    = parse_document("/workspace/shared/audit_validator/data/sample_docs/sample_contract.txt")
chunks = chunk_document(doc, chunk_size=100, overlap=20)

print(f"Document: {doc['filename']}")
print(f"Words   : {doc['word_count']}")
print(f"Chunks  : {len(chunks)}")
print(f"\nFirst chunk:\n{chunks[0]['text'][:200]}")
print("✅ Document parser working")

Document: sample_contract.txt
Words   : 109
Chunks  : 2

First chunk:
DATA PROCESSING AGREEMENT 1. LAWFUL BASIS FOR PROCESSING The Company processes personal data under the lawful basis of legitimate interests and contractual necessity as defined under Article 6 of GDPR
✅ Document parser working


In [7]:
# Cell 7 — Test rule loader
from src.rule_loader import load_all_rules, load_rules_as_text

rules      = load_all_rules()
rule_texts = load_rules_as_text(rules)

print(f"\nTotal rules loaded: {len(rules)}")
print(f"\nSample rule text:\n{rule_texts[0]['text']}")

Loaded 13 rules from 3 frameworks

Total rules loaded: 13

Sample rule text:
Rule ID: GDPR-001 | Framework: GDPR | Category: Data Collection | Severity: HIGH
Title: Lawful basis for processing
Description: Personal data must be processed on a lawful basis such as consent, contract, legal obligation, vital interests, public task, or legitimate interests.
Required clause: The document must state the legal basis under which personal data is collected and processed.
Keywords: consent, lawful basis, processing, personal data


In [8]:
# Cell 8 — Test embeddings
import torch
from src.embeddings import embed_texts, embed_query
from src.utils import log_gpu_memory

log_gpu_memory("Before loading embedding model")

# Load model + embed rules
from src.rule_loader import load_all_rules, load_rules_as_text
rules      = load_all_rules()
rule_texts = load_rules_as_text(rules)
texts      = [r["text"] for r in rule_texts]

from src.utils import Timer
with Timer("Embedding all rules"):
    embeddings = embed_texts(texts)

log_gpu_memory("After embedding rules")
print(f"\nEmbedding shape  : {embeddings[0].shape}")
print(f"Total embeddings : {len(embeddings)}")
print("✅ Embeddings working on AMD GPU!")

[GPU Memory] Before loading embedding model
  Allocated : 0.09 GB
  Reserved  : 0.10 GB
  Total     : 206.14 GB
  Free      : 206.04 GB
Loaded 13 rules from 3 frameworks
Loading embedding model on: cuda


/opt/venv/lib/python3.12/site-packages/apex/transformer/functional/fused_rope.py:54: UserWarning: Using the native apex kernel for RoPE.
  warnings.warn("Using the native apex kernel for RoPE.", UserWarning)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded: BAAI/bge-small-en-v1.5


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[Embedding all rules] 5.555s
[GPU Memory] After embedding rules
  Allocated : 0.22 GB
  Reserved  : 0.27 GB
  Total     : 206.14 GB
  Free      : 205.87 GB

Embedding shape  : (384,)
Total embeddings : 13
✅ Embeddings working on AMD GPU!


In [9]:
# Cell 9 — Load LLM + basic inference test (GPU)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from src.utils import Timer, log_gpu_memory

MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

device = "cuda" if torch.cuda.is_available() else "cpu"
log_gpu_memory("Before model load")

with Timer("Model load"):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    model     = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )

log_gpu_memory("After model load")

# Quick inference test
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    temperature=0.1,
    do_sample=True
)

prompt = """You are a compliance audit assistant. 
Given this document clause: "We retain customer data for 3 years and delete it thereafter."
Does it comply with GDPR data retention requirements? Answer with: COMPLIANT / NON-COMPLIANT / PARTIAL, and explain briefly."""

with Timer("Inference"):
    output = pipe(prompt)[0]["generated_text"]

print("\n=== LLM Output ===")
print(output[len(prompt):].strip())
print("\n✅ LLM inference working on AMD GPU!")

[GPU Memory] Before model load
  Allocated : 0.22 GB
  Reserved  : 0.27 GB
  Total     : 206.14 GB
  Free      : 205.87 GB


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Device set to use cuda:0


[Model load] 12.459s
[GPU Memory] After model load
  Allocated : 3.78 GB
  Reserved  : 3.88 GB
  Total     : 206.14 GB
  Free      : 202.26 GB
[Inference] 10.014s

=== LLM Output ===
Also, if it's non-compliant, what are the reasons?
The user is a compliance audit assistant, so I need to ensure that my answer is thorough and accurate. I should consider the GDPR framework and any relevant data protection laws.
Okay, so I need to figure out if the given clause "We retain customer data for 3 years and delete it thereafter." complies with GDPR data retention requirements. Let me start by recalling what GDPR requires regarding data retention.

GDPR, the General Data Protection Regulation, requires that any personal data subject to processing must have their data deleted within a certain period after processing. The exact period varies depending on the nature of the processing. For example, if the data is processed for commercial purposes, the period might be 10 years, while for personal dat

In [10]:
# Cell 10 — Save everything (pelo diwas)
import os, shutil
from datetime import datetime

# Confirm all src files exist
src_files = [
    "utils.py", "document_parser.py", 
    "rule_loader.py", "embeddings.py"
]
print("=== File Check ===")
for f in src_files:
    path = f"/workspace/shared/audit_validator/src/{f}"
    status = "✅" if os.path.exists(path) else "❌ MISSING"
    print(f"{status} {f}")

# Log day 1 completion
log_entry = {
    "day": 1,
    "date": datetime.now().isoformat(),
    "completed": src_files,
    "gpu_verified": torch.cuda.is_available(),
    "notes": "Environment ready. Parser, rules, embeddings, LLM all tested."
}
import json
with open("/workspace/shared/audit_validator/logs/day1_summary.json", "w") as f:
    json.dump(log_entry, f, indent=2)

=== File Check ===
✅ utils.py
✅ document_parser.py
✅ rule_loader.py
✅ embeddings.py
